In [1]:
import random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)
device = "cuda" if torch.cuda.is_available() else "mps"

In [2]:
import numpy as np

def build_time_index(time_min: np.ndarray):
    time_min = np.asarray(time_min).astype(np.int64)
    return {int(t): i for i, t in enumerate(time_min)}

def align_by_time_min(
    base_time_min: np.ndarray,
    query_time_min: np.ndarray,
    *,
    name="align",
    require_sorted=False,
):
    """
    Returns:
      idx_base: indices into base_time_min for matched times
      idx_query: indices into query_time_min for matched times
      report: dict with counts + missing times
    """
    base_time_min = np.asarray(base_time_min).astype(np.int64)
    query_time_min = np.asarray(query_time_min).astype(np.int64)

    if require_sorted:
        if np.any(np.diff(base_time_min) < 0):
            print(f"WARNING[{name}] base_time_min not sorted")
        if np.any(np.diff(query_time_min) < 0):
            print(f"WARNING[{name}] query_time_min not sorted")

    base_map = build_time_index(base_time_min)

    idx_base = []
    idx_query = []
    missing_in_base = 0
    for j, t in enumerate(query_time_min):
        i = base_map.get(int(t), None)
        if i is None:
            missing_in_base += 1
            continue
        idx_base.append(i)
        idx_query.append(j)

    idx_base = np.asarray(idx_base, dtype=np.int64)
    idx_query = np.asarray(idx_query, dtype=np.int64)

    report = {
        "n_base": int(base_time_min.shape[0]),
        "n_query": int(query_time_min.shape[0]),
        "n_matched": int(idx_base.shape[0]),
        "missing_in_base": int(missing_in_base),
    }
    if missing_in_base > 0:
        print(f"WARNING[{name}] {missing_in_base} query times not found in base_time_min")

    return idx_base, idx_query, report


In [3]:
def inject_synthetic_night_csi_into_combined(
    combined_npz,
    band_npz,
    *,
    combined_time_key="time_min",
    combined_csi_key="csi",
    band_time_key="time_min",
    band_x_key="band",              # change to your real key
    band_is_night_key=None,         # optional: bool mask in band_npz
    night_value_rule="y_eq_0",      # "y_eq_0" or "mask_key"
    predict_fn=None,                # REQUIRED: function(band_x)->pred_csi
    verbose=True,
):
    """
    Returns a new dict ready for np.savez with same time axis/length as combined.
    Replaces night CSI at matched timestamps with predictions.

    night_value_rule:
      - "y_eq_0": treat night frames in combined as those where all pixels == 0
      - "mask_key": use band_npz[band_is_night_key] to select which band rows to predict/insert
    """
    if predict_fn is None:
        raise ValueError("predict_fn is required (band_x -> predicted_csi)")

    ct = np.asarray(combined_npz[combined_time_key]).astype(np.int64)
    csi = np.asarray(combined_npz[combined_csi_key])
    bt = np.asarray(band_npz[band_time_key]).astype(np.int64)
    bx = np.asarray(band_npz[band_x_key])

    # align by time
    idx_c, idx_b, rep = align_by_time_min(ct, bt, name="combined<-band")
    if verbose:
        print("align report:", rep)

    # decide which rows are "night" for replacement
    if night_value_rule == "y_eq_0":
        # night if all pixels are zero at that timestamp (per-frame)
        # csi shape either (T,H,W) or (T,1,H,W) or (T,C,H,W) - handle generically
        csi_frame = csi
        # reduce over non-time dims
        night_mask_combined = (np.nan_to_num(csi_frame[idx_c]) == 0).all(axis=tuple(range(1, csi_frame.ndim)))
    elif night_value_rule == "mask_key":
        if band_is_night_key is None:
            raise ValueError("band_is_night_key required when night_value_rule='mask_key'")
        night_mask_combined = np.asarray(band_npz[band_is_night_key])[idx_b].astype(bool)
    else:
        raise ValueError("night_value_rule must be 'y_eq_0' or 'mask_key'")

    # select band rows to predict and corresponding combined indices
    idx_c_night = idx_c[night_mask_combined]
    idx_b_night = idx_b[night_mask_combined]

    if verbose:
        print(f"n_matched={len(idx_c)}, n_night_to_replace={len(idx_c_night)}")

    # predict CSI for those band rows
    bx_night = bx[idx_b_night]
    pred = predict_fn(bx_night)  # expected shape (N,H,W) or (N,1,H,W)

    pred = np.asarray(pred)
    # normalize prediction to match combined CSI frame shape per timestep
    # combined csi can be (T,H,W) or (T,1,H,W). We'll support both.
    if csi.ndim == 3:   # (T,H,W)
        if pred.ndim == 4 and pred.shape[1] == 1:
            pred = pred[:, 0]
        if pred.ndim != 3:
            raise ValueError(f"pred must be (N,H,W) to fit combined (T,H,W), got {pred.shape}")
    elif csi.ndim == 4: # (T,1,H,W) or (T,C,H,W)
        if pred.ndim == 3:
            pred = pred[:, None, :, :]
        if pred.ndim != 4:
            raise ValueError(f"pred must be (N,C,H,W) to fit combined (T,C,H,W), got {pred.shape}")
        if pred.shape[1] != csi.shape[1]:
            # allow pred (N,1,H,W) into csi (T,1,H,W) only
            raise ValueError(f"channel mismatch: pred C={pred.shape[1]} vs combined C={csi.shape[1]}")
    else:
        raise ValueError(f"combined csi unsupported ndim={csi.ndim}")

    # inject
    csi_new = np.array(csi, copy=True)
    csi_new[idx_c_night] = pred

    out = dict(combined_npz)
    out[combined_csi_key] = csi_new
    out["report_inject"] = {
        **rep,
        "n_night_replaced": int(len(idx_c_night)),
        "night_value_rule": night_value_rule,
    }
    return out


In [4]:
combined =np.load("Data/CAMS/modeldata/combined_cams_data_no_syn.npz", allow_pickle=True)

In [9]:
combined["data"].shape

(17498, 2, 50, 50)

In [ ]:
combined["time_min"] #epoach minutes

array([26824320, 26824380, 26824440, ..., 27875340, 27875400, 27875460],
      shape=(17498,))

In [11]:
from src.model_Twin import TwinFiLMUNetTiny

In [16]:
# how to get model prediciton from bands
device = torch.device('mps')
import torch
import torch.nn.functional as F
model = TwinFiLMUNetTiny(feature_size=48).to(device)
device = torch.device("mps" if torch.backends.mps.is_available()
                      else "cuda" if torch.cuda.is_available()
                      else "cpu")

# 1) load checkpoint
ckpt = torch.load("Model_checkpoint/ckpt_last.pt", map_location=device)
model.load_state_dict(ckpt["model"])
model.to(device)

TwinFiLMUNetTiny(
  (enc_atmos): Sequential(
    (0): Conv2d(9, 48, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): GroupNorm(8, 48, eps=1e-05, affine=True)
    (2): GELU(approximate='none')
    (3): ResBlockGN(
      (c1): Conv2d(48, 48, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (n1): GroupNorm(8, 48, eps=1e-05, affine=True)
      (c2): Conv2d(48, 48, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (n2): GroupNorm(8, 48, eps=1e-05, affine=True)
      (act): GELU(approximate='none')
    )
  )
  (enc_geo): Sequential(
    (0): Conv2d(3, 48, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): GroupNorm(8, 48, eps=1e-05, affine=True)
    (2): GELU(approximate='none')
  )
  (film): FiLM(
    (gamma): Conv2d(48, 48, kernel_size=(1, 1), stride=(1, 1))
    (beta): Conv2d(48, 48, kernel_size=(1, 1), stride=(1, 1))
  )
  (down): Sequential(
    (0): Conv2d(48, 96, kernel_size=(3, 3), stride=(2, 2), paddi

In [19]:

night = np.load("Data/syntheticCSI/csi_synthetic_hourly_night.npz", allow_pickle=True)
night['csi'].shape

(10068, 1, 50, 50)

In [40]:
rawnightband = np.load("Data/bands/modeldata/bands_night_paired_twillight.npz", allow_pickle=True)
rawnightband["data"].shape

(10068, 15, 50, 50)

In [22]:
night["time_hourly"]

array([26824860, 27196440, 27196500, ..., 27236710, 27251110, 27249670],
      shape=(10068,))

In [27]:
import numpy as np

def _dedup_keep_last_by_time(t, arr=None):
    t = np.asarray(t).astype(np.int64)
    order = np.argsort(t, kind="mergesort")  # stable
    ts = t[order]
    a = np.asarray(arr)[order] if arr is not None else None

    # keep last occurrence of each time
    is_last = np.r_[ts[1:] != ts[:-1], True]
    keep = np.where(is_last)[0]
    t_u = ts[keep]
    a_u = a[keep] if a is not None else None
    rep = {"n_in": int(t.size), "n_out": int(t_u.size), "duplicates_dropped": int(t.size - t_u.size)}
    return t_u, a_u, rep

def _warn_time_gaps(t, *, name, expected_dt=60, gap_factor=2.5):
    t = np.asarray(t).astype(np.int64)
    if t.size < 3:
        return
    ts = np.sort(t)
    dt = np.diff(ts)
    dt_pos = dt[dt > 0]
    med = float(np.median(dt_pos)) if dt_pos.size else float("nan")
    thr = max(int(expected_dt * gap_factor), int((med if np.isfinite(med) else expected_dt) * gap_factor))
    bad = np.where(dt > thr)[0]
    for k in bad[:10]:
        print(f"WARNING[{name}] gap: {ts[k]} -> {ts[k+1]} (dt={dt[k]}, median_dt~{med})")
    if bad.size > 10:
        print(f"WARNING[{name}] {bad.size} gaps found (showing first 10)")

def build_csi_and_replace_night_by_time(
    combined_npz_path: str,
    synthetic_npz_path: str,
    out_npz_path: str,
    *,
    combined_time_key="time_min",
    combined_data_key="data",     # (T,2,H,W) [0]=GHI, [1]=CSGHI
    syn_csi_key="csi",            # (N,1,H,W)
    syn_time_key=None,            # auto: "time_min" or "time_hourly"
    night_rule="csi_eq_0",        # detect night on computed CSI
    night_eps=0.0,                # use >0 if night is near-zero not exact
    csi_clip=(0.0, 1.0),
    expected_dt=60,
    gap_factor=2.5,
    verbose=True,
):
    """
    Output NPZ:
      - time_min: (T,)
      - csi: (T,1,H,W)  where night frames are replaced by synthetic CSI when time matches

    Notes:
      - Dedups combined and synthetic by time (keep last)
      - Replacement only for frames that are "night" in computed combined CSI
      - Uses combined time axis as the final time axis
    """
    comb = np.load(combined_npz_path, allow_pickle=True)
    syn = np.load(synthetic_npz_path, allow_pickle=True)

    if syn_time_key is None:
        if "time_min" in syn.files:
            syn_time_key = "time_min"
        elif "time_hourly" in syn.files:
            syn_time_key = "time_hourly"
        else:
            raise KeyError(f"Cannot find synthetic time key. Available: {syn.files}")

    t_c = np.asarray(comb[combined_time_key]).astype(np.int64)
    data = np.asarray(comb[combined_data_key])
    if data.ndim != 4 or data.shape[1] != 2:
        raise ValueError(f"combined data must be (T,2,H,W), got {data.shape}")

    ghi = data[:, 0]   # (T,H,W)
    cs  = data[:, 1]   # (T,H,W)

    # computed CSI (safe)
    with np.errstate(divide="ignore", invalid="ignore"):
        csi = np.where(cs > 0, ghi / cs, 0.0).astype(np.float32)
    lo, hi = csi_clip
    csi = np.clip(csi, lo, hi)                 # (T,H,W)
    csi = csi[:, None, :, :]                   # (T,1,H,W)

    t_s = np.asarray(syn[syn_time_key]).astype(np.int64)
    csi_syn = np.asarray(syn[syn_csi_key]).astype(np.float32)
    if csi_syn.ndim != 4 or csi_syn.shape[1] != 1:
        raise ValueError(f"synthetic csi must be (N,1,H,W), got {csi_syn.shape}")
    csi_syn = np.clip(csi_syn, lo, hi)

    # dedup both by time
    t_c_u, csi_u, rep_c = _dedup_keep_last_by_time(t_c, csi)
    t_s_u, csi_s_u, rep_s = _dedup_keep_last_by_time(t_s, csi_syn)

    if verbose:
        if rep_c["duplicates_dropped"] > 0:
            print(f"WARNING[combined] dropped {rep_c['duplicates_dropped']} duplicate times (keep last)")
        if rep_s["duplicates_dropped"] > 0:
            print(f"WARNING[synthetic] dropped {rep_s['duplicates_dropped']} duplicate times (keep last)")
        _warn_time_gaps(t_c_u, name="combined", expected_dt=expected_dt, gap_factor=gap_factor)
        _warn_time_gaps(t_s_u, name="synthetic", expected_dt=expected_dt, gap_factor=gap_factor)

    # night mask on combined CSI
    csi_u_hw = csi_u[:, 0]  # (T,H,W)
    if night_rule == "csi_eq_0":
        if night_eps == 0.0:
            night = (csi_u_hw == 0).all(axis=(1, 2))
        else:
            night = (np.abs(csi_u_hw) <= night_eps).all(axis=(1, 2))
    else:
        raise ValueError("Only night_rule='csi_eq_0' supported")

    # align synthetic -> combined using time intersection
    common_t, idx_c, idx_s = np.intersect1d(t_c_u, t_s_u, return_indices=True)

    if verbose:
        miss_in_syn = int(t_c_u.size - common_t.size)
        miss_in_comb = int(t_s_u.size - common_t.size)
        if miss_in_syn > 0:
            print(f"WARNING[align] {miss_in_syn} combined times not found in synthetic")
        if miss_in_comb > 0:
            print(f"WARNING[align] {miss_in_comb} synthetic times not found in combined")

    # only replace where combined is night
    night_on_matched = night[idx_c]
    idx_c_night = idx_c[night_on_matched]
    idx_s_night = idx_s[night_on_matched]

    csi_out = np.array(csi_u, copy=True)
    csi_out[idx_c_night] = csi_s_u[idx_s_night]

    report = {
        "combined_path": combined_npz_path,
        "synthetic_path": synthetic_npz_path,
        "syn_time_key": syn_time_key,
        "n_combined_in": int(t_c.size),
        "n_combined_dedup": int(t_c_u.size),
        "n_synthetic_in": int(t_s.size),
        "n_synthetic_dedup": int(t_s_u.size),
        "n_time_matched": int(common_t.size),
        "n_night_in_combined": int(night.sum()),
        "n_night_replaced": int(idx_c_night.size),
    }

    np.savez_compressed(out_npz_path, time_min=t_c_u, csi=csi_out, report=report)
    if verbose:
        print("saved:", out_npz_path)
        print("out shapes:", csi_out.shape, t_c_u.shape)
        print("report:", report)

    return {"time_min": t_c_u, "csi": csi_out, "report": report}


In [28]:
out = build_csi_and_replace_night_by_time(
    combined_npz_path="Data/CAMS/modeldata/combined_cams_data_no_syn.npz",
    synthetic_npz_path="Data/syntheticCSI/csi_synthetic_hourly_night.npz",
    out_npz_path="Data/syntheticCSI/combined_csi_with_syn_night.npz",
    syn_time_key="time_hourly",   # or leave None
    night_eps=0.0,                # set 1e-6 if needed
    verbose=True,
)


WARNING[combined] gap: 27444360 -> 27444720 (dt=360, median_dt~60.0)
WARNING[combined] gap: 27628380 -> 27629280 (dt=900, median_dt~60.0)
WARNING[synthetic] gap: 26825750 -> 26826300 (dt=550, median_dt~60.0)
WARNING[synthetic] gap: 26827190 -> 26827740 (dt=550, median_dt~60.0)
WARNING[synthetic] gap: 26828630 -> 26829180 (dt=550, median_dt~60.0)
WARNING[synthetic] gap: 26830070 -> 26830620 (dt=550, median_dt~60.0)
WARNING[synthetic] gap: 26831510 -> 26832060 (dt=550, median_dt~60.0)
WARNING[synthetic] gap: 26832950 -> 26833500 (dt=550, median_dt~60.0)
WARNING[synthetic] gap: 26834390 -> 26834950 (dt=560, median_dt~60.0)
WARNING[synthetic] gap: 26835830 -> 26836390 (dt=560, median_dt~60.0)
WARNING[synthetic] gap: 26837270 -> 26837830 (dt=560, median_dt~60.0)
WARNING[synthetic] gap: 26838710 -> 26839270 (dt=560, median_dt~60.0)
WARNING[synthetic] 729 gaps found (showing first 10)
WARNING[align] 7802 combined times not found in synthetic
WARNING[align] 372 synthetic times not found in com

In [33]:
out_npz_path="Data/syntheticCSI/combined_csi_with_syn_night.npz"
syn = np.load(out_npz_path,allow_pickle=True)

In [34]:
syn.keys()

KeysView(NpzFile 'Data/syntheticCSI/combined_csi_with_syn_night.npz' with keys: time_min, csi, report)

In [38]:
syn["csi"].shape

(17498, 1, 50, 50)

In [39]:
syn["time_min"]

array([26824320, 26824380, 26824440, ..., 27875340, 27875400, 27875460],
      shape=(17498,))

In [42]:
import numpy as np

comb = np.load("Data/CAMS/modeldata/combined_cams_data_no_syn.npz", allow_pickle=True)
syn  = np.load("Data/syntheticCSI/csi_synthetic_hourly_night.npz", allow_pickle=True)

t_c = np.asarray(comb["time_min"], dtype=np.int64)
t_s = np.asarray(syn["time_hourly"], dtype=np.int64)

t_c_sorted = np.sort(t_c)
cset = set(t_c.tolist())

diffs = []
no_cand = 0
checked = 0

for t in t_s[:5000]:
    checked += 1
    if int(t) in cset:
        continue

    j = np.searchsorted(t_c_sorted, t)
    cand = []
    if j > 0:
        cand.append(abs(int(t) - int(t_c_sorted[j-1])))
    if j < t_c_sorted.size:
        cand.append(abs(int(t) - int(t_c_sorted[j])))
    if not cand:
        no_cand += 1
        continue
    diffs.append(min(cand))

print("checked:", checked)
print("exact_matches:", checked - len(diffs) - no_cand)
print("no_candidate_cases:", no_cand)
print("non_matching_with_distance:", len(diffs))

if len(diffs) == 0:
    print("No non-matching timestamps found (or no valid distances).")
else:
    diffs = np.asarray(diffs)
    print("median nearest-minute diff:", float(np.median(diffs)))
    print("p95 nearest-minute diff:", float(np.percentile(diffs, 95)))
    print("min/max diff:", int(diffs.min()), int(diffs.max()))


checked: 5000
exact_matches: 5000
no_candidate_cases: 0
non_matching_with_distance: 0
No non-matching timestamps found (or no valid distances).
